# Реализация кода из статьи Google [How Important Is a Neuron?](https://arxiv.org/abs/1805.12233)

In [1]:
# %%
import math
from collections import OrderedDict

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from torch.func import functional_call, jvp

torch.set_grad_enabled(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
DTYPE = torch.float32

print("device:", DEVICE)
print("torch:", torch.__version__)

device: mps
torch: 2.9.1


In [2]:
# %%
# Загрузка именно классификатора YOLO11s
yolo = YOLO("yolo11s-cls.pt")
model = yolo.model.to(DEVICE).eval()

# Имена классов ImageNet
class_names = yolo.names

print(type(model).__name__)
print("num classes:", len(class_names))
print("first 10 classes:", {k: class_names[k] for k in range(10)})

ClassificationModel
num classes: 1000
first 10 classes: {0: 'tench', 1: 'goldfish', 2: 'great_white_shark', 3: 'tiger_shark', 4: 'hammerhead', 5: 'electric_ray', 6: 'stingray', 7: 'cock', 8: 'hen', 9: 'ostrich'}


In [3]:
# %%
# Параметры изображения для YOLO11-cls:
# classification-чекпойнты Ultralytics для YOLO11 используют 224x224.
IMG_SIZE = 224

def load_image(path, img_size=IMG_SIZE):
    img = Image.open(path).convert("RGB")
    img = img.resize((img_size, img_size))
    img_np = np.asarray(img).astype(np.float32) / 255.0
    x = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE, DTYPE)
    return x, img_np

def black_baseline_like(x):
    return torch.zeros_like(x)

IMAGE_PATH = "data/jess.jpg"  # <- замени на своё изображение
x, img_np = load_image(IMAGE_PATH)
x0 = black_baseline_like(x)

print("x shape:", tuple(x.shape))

x shape: (1, 3, 224, 224)


In [4]:
# %%
# Список candidate layers: обычно имеет смысл выбирать conv/feature layers,
# а не финальный classifier head.
def list_named_modules(model):
    return [(name, module) for name, module in model.named_modules() if name != ""]

candidate_layers = []
for name, module in list_named_modules(model):
    if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d, torch.nn.SiLU, torch.nn.ReLU, torch.nn.Sequential)):
        candidate_layers.append((name, type(module).__name__))

for i, (name, typ) in enumerate(candidate_layers):
    print(f"{i:2d}: {name:50s} {typ}")

 0: model                                              Sequential
 1: model.0.conv                                       Conv2d
 2: model.0.bn                                         BatchNorm2d
 3: model.0.act                                        SiLU
 4: model.1.conv                                       Conv2d
 5: model.1.bn                                         BatchNorm2d
 6: model.2.cv1.conv                                   Conv2d
 7: model.2.cv1.bn                                     BatchNorm2d
 8: model.2.cv2.conv                                   Conv2d
 9: model.2.cv2.bn                                     BatchNorm2d
10: model.2.m.0.cv1.conv                               Conv2d
11: model.2.m.0.cv1.bn                                 BatchNorm2d
12: model.2.m.0.cv2.conv                               Conv2d
13: model.2.m.0.cv2.bn                                 BatchNorm2d
14: model.3.conv                                       Conv2d
15: model.3.bn                        

In [5]:
# %%
# Выбери слой вручную из напечатанного списка.
# Для conductance лучше брать слой, у которого activation имеет spatial shape [B, C, H, W].
#LAYER_NAME = candidate_layers[80][0]  # <- при необходимости замени вручную
LAYER_NAME = "model.9"
print("chosen layer:", LAYER_NAME)

chosen layer: model.9


In [12]:
# %%
# Обычный forward hook без functional_call
layer_store = {}

def make_hook(name):
    def hook(module, inp, out):
        layer_store[name] = out
    return hook

target_module = dict(model.named_modules())[LAYER_NAME]
hook_handle = target_module.register_forward_hook(make_hook(LAYER_NAME))

def forward_with_layer(x_in):
    layer_store.clear()
    logits = model(x_in)
    act = layer_store[LAYER_NAME]
    return logits, act

# %%
with torch.no_grad():
    out, act = forward_with_layer(x)
    if isinstance(out, (tuple, list)):
        probs, logits = out
    else:
        logits = out
    target_class = int(logits[0].argmax().item())
    target_name = class_names[target_class]
    target_logit = float(logits[0, target_class].item())

print("target class:", target_class, target_name)
print("target logit:", target_logit)
print("layer activation shape:", tuple(act.shape))

target class: 171 Italian_greyhound
target logit: 11.263603210449219
layer activation shape: (1, 512, 7, 7)


In [13]:
# %%
# Conductance по статье для выбранного hidden layer.
#
# Математика:
# x(α) = x0 + α (x - x0)
# F(x(α)) = logit целевого класса
# y(α) = activation выбранного слоя
#
# Conductance:
#   Cond(y) = ∫_0^1 [ ∂F/∂y (x(α)) · dy/dα (α) ] dα
#
# Здесь:
# - grad_y = ∂F/∂y
# - act_tangent = dy/dα
# - integrand = grad_y * act_tangent
#
# Численное интегрирование делаем midpoint rule.

import gc

def clear_backend_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def compute_conductance(
    x,
    x0,
    target_class,
    n_steps=64,
    fd_eps=1e-3,
    clear_every=8,
):
    x = x.contiguous()
    x0 = x0.contiguous()
    delta_x = (x - x0).contiguous()

    alphas = torch.linspace(0.0, 1.0, n_steps + 1, device=x.device, dtype=x.dtype)
    step = 1.0 / n_steps

    cond_accum = None
    used_fallback = False

    for k in range(n_steps):
        alpha = (alphas[k] + alphas[k + 1]) / 2.0

        x_alpha = (x0 + alpha * delta_x).contiguous().detach().requires_grad_(True)

        out, act = forward_with_layer(x_alpha)
        if isinstance(out, (tuple, list)):
            probs, logits = out
        else:
            logits = out

        score = logits[0, target_class]

        # ∂F/∂y
        grad_y = torch.autograd.grad(score, act, retain_graph=False, create_graph=False)[0]

        try:
            def act_only(inp):
                inp = inp.contiguous()
                _, a = forward_with_layer(inp)
                return a

            # dy/dα
            _, act_tangent = jvp(act_only, (x_alpha,), (delta_x,))

        except RuntimeError as e:
            if "view size is not compatible with input tensor's size and stride" not in str(e):
                raise

            used_fallback = True

            alpha_plus = min(float(alpha.item()) + fd_eps, 1.0)
            alpha_minus = max(float(alpha.item()) - fd_eps, 0.0)
            denom = alpha_plus - alpha_minus

            if denom == 0.0:
                raise RuntimeError("Не удалось построить finite-difference fallback для dy/dalpha.")

            with torch.no_grad():
                _, act_plus = forward_with_layer((x0 + alpha_plus * delta_x).contiguous())
                _, act_minus = forward_with_layer((x0 + alpha_minus * delta_x).contiguous())

            # dy/dα ≈ [y(α+ε) - y(α-ε)] / (2ε)
            act_tangent = (act_plus - act_minus) / denom

            del act_plus, act_minus

        integrand = grad_y * act_tangent

        if cond_accum is None:
            cond_accum = integrand.detach()
        else:
            cond_accum = cond_accum + integrand.detach()

        del x_alpha, out, act, logits, score, grad_y, act_tangent, integrand
        if 'probs' in locals():
            del probs
        layer_store.clear()

        if (k + 1) % clear_every == 0:
            clear_backend_cache()

    cond = cond_accum * step

    clear_backend_cache()

    if used_fallback:
        print("Предупреждение: jvp на текущем backend не сработал, для dy/dalpha использована конечная разность.")

    return cond.detach()


def reduce_filter_scores(cond_tensor):
    if cond_tensor.ndim < 2:
        raise ValueError(
            f"Ожидался tensor с batch-осью и хотя бы одной feature-осью, получено shape={tuple(cond_tensor.shape)}"
        )

    per_sample = cond_tensor[0]

    if per_sample.ndim == 1:
        return per_sample

    reduce_dims = tuple(range(1, per_sample.ndim))
    return per_sample.sum(dim=reduce_dims)


cond_tensor = compute_conductance(
    x=x,
    x0=x0,
    target_class=target_class,
    n_steps=64,
)

print("cond tensor shape:", tuple(cond_tensor.shape))
print("filter_scores shape:", tuple(reduce_filter_scores(cond_tensor).shape))

cond tensor shape: (1, 512, 7, 7)
filter_scores shape: (512,)


In [14]:
# %%
# Агрегация conductance для слоя любой размерности

filter_scores = reduce_filter_scores(cond_tensor)
layer_score = cond_tensor.sum()

topk = min(10, filter_scores.numel())
top_vals, top_idx = torch.topk(filter_scores.abs(), k=topk)

print("top filters by |conductance|:")
for rank, idx in enumerate(top_idx.tolist(), start=1):
    val = float(filter_scores[idx].item())
    print(f"{rank:2d}. filter {idx:4d}: {val:+.6f}")

print("\nlayer conductance sum:", float(layer_score.item()))
print("example neuron conductance:", float(cond_tensor.reshape(-1)[0].item()))
print("example filter conductance:", float(filter_scores[0].item()))

top filters by |conductance|:
 1. filter  309: +1.152036
 2. filter  156: +0.865820
 3. filter  255: +0.795515
 4. filter  413: +0.794366
 5. filter  490: -0.767644
 6. filter  124: +0.601888
 7. filter  299: +0.538830
 8. filter   93: +0.502713
 9. filter  378: +0.486036
10. filter  106: -0.462150

layer conductance sum: 12.002995491027832
example neuron conductance: -0.00037865291233174503
example filter conductance: 0.012485548853874207


In [15]:
# %%
def calculate_abs_error(x, x0=x0, target_class=target_class):
    with torch.no_grad():
        out_x, _ = forward_with_layer(x)
        out_x0, _ = forward_with_layer(x0)

        if isinstance(out_x, (tuple, list)):
            probs_x, logits_x = out_x
        else:
            logits_x = out_x

        if isinstance(out_x0, (tuple, list)):
            probs_x0, logits_x0 = out_x0
        else:
            logits_x0 = out_x0

        fx = float(logits_x[0, target_class].item())
        fx0 = float(logits_x0[0, target_class].item())

    print("F(x)            =", fx)
    print("F(x0)           =", fx0)
    print("F(x) - F(x0)    =", fx - fx0)
    print("sum conductance =", float(layer_score.item()))
    print("abs error       =", abs((fx - fx0) - float(layer_score.item())))

calculate_abs_error(x)

F(x)            = 11.263603210449219
F(x0)           = -0.974008321762085
F(x) - F(x0)    = 12.237611532211304
sum conductance = 12.002995491027832
abs error       = 0.23461604118347168


In [10]:
# %%
hook_handle.remove()